# 00 — Ingesta automática del año 2026

## Objetivo

Este notebook sirve como guion técnico y visual para la exposición. Todas las
comprobaciones usan los artefactos completos del proyecto; las vistas de pocas filas
son únicamente para mostrar el esquema, no son el conjunto usado por el pipeline.

**QUÉ EXPLICAR:** primero diga qué problema resuelve esta etapa, después muestre la
evidencia y termine conectándola con la siguiente capa.

## Preparación

Ejecútelo desde el contenedor `spark` con la raíz `/workspace`. La celda también
encuentra el repositorio cuando el notebook se abre desde la carpeta local.

In [ ]:
from pathlib import Path
import json

# Encontrar la raíz permite usar el mismo notebook en Docker o en el repositorio local.
candidates = [Path.cwd(), Path.cwd().parent, Path('/workspace')]
ROOT = next(
    path for path in candidates
    if (path / 'config' / 'pipeline.yaml').exists()
)
DATA = ROOT / 'data'
EXPORTS = ROOT / 'exports'
print(f'Raíz del proyecto: {ROOT}')

## Pasos

### 1. Alcance seguro de la ejecución

Bronze y Silver reciben `--years 2026`. Gold se recalcula desde Silver completo para que
los dashboards mantengan 2023–2026. No se usa `--force`, por lo que los archivos válidos
ya descargados se omiten de forma idempotente.

**QUÉ EXPLICAR:** el año corriente se descubre en la página oficial; no se codifican meses.

In [ ]:
# Modo seguro: esta celda solo muestra los comandos de la exposición.
# Cambie a True únicamente cuando quiera ejecutar de verdad dentro del contenedor Spark.
EJECUTAR = False
comandos = [
    ('tlc-pipeline catalog --years 2026 --catalog-output '
     '/workspace/data/bronze/_catalog_2026.json'),
    ('tlc-pipeline ingest --years 2026 --catalog-input '
     '/workspace/data/bronze/_catalog_2026.json'),
    'tlc-pipeline silver --years 2026',
    'tlc-pipeline gold',
    'tlc-pipeline models',
    'tlc-pipeline audit-export',
    'tlc-pipeline powerbi',
    'tlc-pipeline verify --powerbi-path /workspace/powerbi',
]
print('\n'.join(comandos))

if EJECUTAR:
    import subprocess
    for comando in comandos:
        print(f'\n>>> {comando}')
        subprocess.run(comando, shell=True, check=True, cwd=ROOT)

### 2. Evidencia de archivos 2026 ya publicados

In [ ]:
import re
from collections import Counter

# El catálogo conserva lo descubierto; contar por servicio demuestra el alcance real.
catalog_path = DATA / 'bronze' / '_catalog.json'
catalog = json.loads(catalog_path.read_text(encoding='utf-8'))
entries = catalog.get('files', catalog) if isinstance(catalog, dict) else catalog
current = [
    item for item in entries
    if int(item['year']) == 2026 and item.get('available', True)
]
print('Archivos 2026 disponibles:', len(current))
print('Por servicio:', dict(Counter(item['service'] for item in current)))
print('Meses:', sorted({int(item['month']) for item in current}))

## Comprobaciones

In [ ]:
# El comando público para la defensa tiene modo Demo por defecto y modo Execute explícito.
script = ROOT / 'scripts' / 'run_2026_only.ps1'
assert script.exists()
text = script.read_text(encoding='utf-8')
assert '--years 2026' in text and 'tlc-pipeline verify' in text
print('Automatización 2026 lista:', script)

## Siguiente paso

Ejecute `./scripts/run_2026_only.ps1` para ensayar sin cambios. Durante la exposición use
`./scripts/run_2026_only.ps1 -Mode Execute`. Después muestre Bronze.